In [ ]:
from __future__ import annotations

from pathlib import Path
from uuid import uuid4
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl

from src.data import IngestionAgent
from src.deployment import AdversarialStressTester, DeploymentHarness
from src.evaluation import EvaluationEngine
from src.features import PurgedEraSplitter
from src.models import ModelOrchestrator
from src.notebook_utils import render_promotion_report
from src.risk import NeutralizationEngine
from src.runner import PromotionRunner

agent = IngestionAgent(project_root / 'data' / 'v5.2')
target_col = agent.get_target_names('train')[0]
custom_engine = EvaluationEngine(
    era_col='era',
    prediction_col='prediction',
    target_col=target_col,
    id_col='id',
    backend='custom',
)
official_engine = EvaluationEngine(
    era_col='era',
    prediction_col='prediction',
    target_col=target_col,
    id_col='id',
    backend='official',
)
neutralizer = NeutralizationEngine(project_root / 'artifacts' / 'cache')
harness = DeploymentHarness()

ModuleNotFoundError: No module named 'src'

In [ ]:
# Research and smoke-check scratchpad.
# Keep ad hoc EDA, anchor-fold trials, and feature experiments in this cell.

train_lazy = agent.scan_dataset('train', include_metadata=True)
print('Train schema columns:', len(train_lazy.collect_schema().names()))
print('Available feature subsets:', agent.feature_subset_names())

In [ ]:
# ---------------------------------------------------------
# PROMOTION RUN CONFIGURATION
# ---------------------------------------------------------
PROMOTION_CONFIG = {
    'dataset': 'train',
    'feature_subset': 'small',
    'cv_splits': 5,
    'cv_purge_buffer': 1,
    'era_limit': 30,
    'model_params': {
        'n_estimators': 120,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_child_samples': 20,
    },
    'gates': {
        'min_mean_corr': -1.0,
        'min_sharpe_corr': -10.0,
        'max_drawdown_corr': 10.0,
    },
    'stress_testing': {
        'noise_std_ratio': 0.05,
        'degradation_threshold': 0.40,
    },
    'parity_eras': 5,
}

In [ ]:
selected_eras = (
    agent.scan_dataset(PROMOTION_CONFIG['dataset'], include_metadata=True)
    .select('era')
    .unique()
    .sort('era')
    .collect()
    .get_column('era')
    .to_list()
)
if PROMOTION_CONFIG['era_limit'] is not None:
    selected_eras = selected_eras[:PROMOTION_CONFIG['era_limit']]

runner = PromotionRunner(
    agent=agent,
    dataset_name=PROMOTION_CONFIG['dataset'],
    feature_subset=PROMOTION_CONFIG['feature_subset'],
    target_column=target_col,
    splitter=PurgedEraSplitter(
        n_splits=PROMOTION_CONFIG['cv_splits'],
        purge_buffer=PROMOTION_CONFIG['cv_purge_buffer'],
    ),
    orchestrator=ModelOrchestrator(
        feature_names=agent.get_feature_names(PROMOTION_CONFIG['feature_subset']),
        target_column=target_col,
        model_library='lightgbm',
        model_params=PROMOTION_CONFIG['model_params'],
        prefer_gpu=True,
    ),
    custom_evaluation_engine=custom_engine,
    official_evaluation_engine=official_engine,
    neutralization_engine=neutralizer,
    stress_tester=AdversarialStressTester(
        custom_engine,
        degradation_threshold=PROMOTION_CONFIG['stress_testing']['degradation_threshold'],
        random_state=7,
    ),
    deployment_harness=harness,
    artifact_dir=project_root / 'artifacts' / 'promotion' / f"run_{uuid4().hex[:8]}" ,
    config_metadata=PROMOTION_CONFIG,
    gate_kwargs=PROMOTION_CONFIG['gates'],
    requirements_path=project_root / 'requirements.txt',
    era_slice=selected_eras,
    parity_era_count=PROMOTION_CONFIG['parity_eras'],
    stress_noise_std_ratio=PROMOTION_CONFIG['stress_testing']['noise_std_ratio'],
)

result = runner.run()
render_promotion_report(result)